# Measure Theory Essentials

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 03.01–03.11 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/09_Advanced_Probability/01_measure_theory_essentials.ipynb)

---

## 🎯 Learning Objective

Learn the measure-theoretic foundations of probability — $\sigma$-algebras, measurable functions, and the Lebesgue integral — needed to read modern ML theory papers.

---

## 📐 Theory

Every earlier probability notebook (`03.01`-`03.11`) quietly assumed you can always ask "what is
$P(X \in A)$?" for any set $A$ you might care about, and that expectation $\mathbb{E}[X]$ always
means "integrate $X$ against its density." Both assumptions can fail once probability spaces get
infinite or continuous enough — and modern ML theory papers (PAC-Bayes bounds, diffusion model
proofs, convergence guarantees) are written in a language that patches exactly these failures.
This notebook is not a measure theory course; it is the minimum vocabulary needed to read that
language without flinching at symbols like $\sigma$-algebra, "a.e.", or $\mathcal{B}(\mathbb{R})$.

### Why not every set can be measured

Suppose you want a "length" function $\mu$ on subsets of $[0,1]$ satisfying three reasonable
properties: $\mu([a,b]) = b-a$; $\mu$ is translation-invariant (shifting a set doesn't change its
length); and $\mu$ is *countably additive* — the length of a countable union of disjoint sets is
the sum of their lengths. The **Vitali set** construction (group real numbers into equivalence
classes under "differ by a rational," pick one representative per class) produces a set $V$ that
provably cannot be assigned *any* value $\mu(V)$ consistent with those three properties:
translating $V$ by every rational in $[-1,1]$ and taking the union recovers a set sandwiched
between $[0,1]$ and $[-1,2]$, so countably many disjoint copies of $\mu(V)$ must sum to something
between $1$ and $3$ — impossible for a single number added to itself infinitely often (it's
either $0$, summing to $0$, or positive, summing to $\infty$). This is a proof that "assign a
translation-invariant length to *every* subset of $\mathbb{R}$" is simply inconsistent. Something
has to give, and measure theory's answer is: give up on measuring *every* subset, and carefully
define which subsets ("measurable sets") get to have a length at all.

### $\sigma$-algebras: the sets you're allowed to ask about

A **$\sigma$-algebra** $\mathcal{F}$ on a set $\Omega$ is a collection of subsets of $\Omega$
(**events**, following `03.01`'s language) satisfying three closure rules: $\Omega \in
\mathcal{F}$; if $A \in \mathcal{F}$ then its complement $A^c \in \mathcal{F}$; and if
$A_1, A_2, \ldots \in \mathcal{F}$ then $\bigcup_i A_i \in \mathcal{F}$. These rules guarantee any
question phrased with "not," "all of these," or "at least one of these," starting from sets in
$\mathcal{F}$, lands back in $\mathcal{F}$ — the "askable" events are closed under exactly the
logical operations probability needs. A **probability space** is a triple $(\Omega,\mathcal{F},P)$:
outcomes $\Omega$, a $\sigma$-algebra $\mathcal{F}$ of events allowed a probability, and a measure
$P:\mathcal{F}\to[0,1]$ with $P(\Omega)=1$ and countable additivity on disjoint events — the
rigorous version of the axioms `03.01` introduced informally. On $\mathbb{R}$, the standard choice
is the **Borel $\sigma$-algebra** $\mathcal{B}(\mathbb{R})$: the smallest $\sigma$-algebra
containing every open interval. It contains every set you'd practically construct (intervals,
countable unions/intersections of those) while excluding pathological sets like the Vitali set.

### Measurable functions: random variables made rigorous

`03.02` defined a random variable informally as "a function from outcomes to numbers." The
rigorous version: $X:\Omega\to\mathbb{R}$ is a **measurable function** (a genuine random variable
on $(\Omega,\mathcal{F},P)$) if for every Borel set $B\in\mathcal{B}(\mathbb{R})$, the preimage
$X^{-1}(B) = \{\omega \in \Omega : X(\omega) \in B\}$ belongs to $\mathcal{F}$. This is exactly
what makes $P(X \in B)$ well-defined for every $B$ you'd care about: you're asking $\mathcal{F}$
for the probability of the event $X^{-1}(B)$, and measurability guarantees $\mathcal{F}$ actually
assigns that event a probability. Every distribution in `03.02`/`03.06` is secretly a measurable
function underneath; those notebooks never needed to say so because, for the concrete
constructions used there, measurability holds automatically.

### The Lebesgue integral: integrating against a measure, not just a $dx$

`02.07`'s Riemann integral slices the *domain* into thin vertical strips and sums
$f(x)\cdot(\text{strip width})$. The **Lebesgue integral** instead slices the *range* into thin
horizontal layers: for $f \ge 0$,

$$\int_\Omega f \, d\mu = \int_0^\infty \mu(\{x : f(x) > t\}) \, dt,$$

summing, over every height $t$, how much of the domain (measured by $\mu$) lies above that
height. Riemann integration requires $f$ to be "not too discontinuous"; Lebesgue integration only
requires $f$ to be measurable — a strictly weaker condition — and it assigns a sensible integral
to functions (like the indicator of the rationals) with no Riemann integral at all. When $\mu$ is
a probability measure $P$, this gives the rigorous definition of **expectation**:
$\mathbb{E}[X] = \int_\Omega X \, dP$, replacing the "$\sum_x xP(x)$" / "$\int xf(x)\,dx$" formulas
from `03.03`, which were really shortcuts valid for the discrete and continuous-density cases.

### Convergence almost everywhere — and why ML papers keep writing "a.s."

A sequence of functions $f_n$ converges to $f$ **almost everywhere (a.e.)** — or, when $\mu=P$ is
a probability measure, **almost surely (a.s.)** — if $f_n(\omega) \to f(\omega)$ for every
$\omega$ *except possibly a set of measure zero*. This is genuinely weaker than convergence at
every point: the *typewriter sequence* $f_n = \mathbb{1}_{[j/2^k,(j+1)/2^k)}$ (intervals that
repeatedly sweep across $[0,1]$, shrinking as $n$ grows, with $n=2^k+j$) converges to $0$ in
$L^1$ (its integral shrinks to $0$) yet fails to converge at *every single point* — every $x$ is
hit by infinitely many $f_n$'s, oscillating forever between $0$ and $1$. We verify that pathology
numerically below, contrasted with a genuinely a.e.- (not everywhere-) convergent example:
$g_n(x) = \mathbb{1}_{[0,1/n]}(x)$ converges to $0$ at every $x>0$ but stays stuck at
$g_n(0)=1$ forever — a single bad point, measure zero, excluded by "a.e." When an ML paper states
an SGD iterate converges "almost surely," it makes exactly this kind of claim: convergence holds
on a set of random seeds with probability $1$, while not ruling out a measure-zero set of
unlucky, non-converging runs — a precise, checkable statement, not hedging.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

The "typewriter sequence" of shrinking, sweeping intervals converges to zero in total area yet
never settles down at any single point — watching its indicator function sweep across $[0,1]$
makes the gap between "converges in integral" and "converges everywhere" visible directly.

In [ ]:
# The "typewriter sequence": f_n = indicator of an interval of width 1/2^k that sweeps left to
# right across [0,1] as n increases, restarting narrower each time k increments (n = 2^k + j).
# This is a standard example separating "convergence in integral" from "convergence at a point."
def typewriter_interval(n):
    k = int(np.floor(np.log2(n)))
    j = n - 2 ** k
    lo, hi = j / 2 ** k, (j + 1) / 2 ** k
    return lo, hi

def typewriter_fn(n, x):
    lo, hi = typewriter_interval(n)
    return np.where((x >= lo) & (x < hi), 1.0, 0.0)

x_grid = np.linspace(0, 1, 2000)
fig, axes = plt.subplots(2, 1, figsize=(9, 8))

# Top: overlay f_n for several n -- the "on" interval visibly shrinks and sweeps
ns_to_plot = [1, 2, 3, 6, 12, 24]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(ns_to_plot)))
for n, color in zip(ns_to_plot, colors):
    lo, hi = typewriter_interval(n)
    axes[0].plot(x_grid, typewriter_fn(n, x_grid) + 0.02 * ns_to_plot.index(n), color=color, lw=2,
                 label=f"n={n}: on [{lo:.3f}, {hi:.3f})")
axes[0].set_title("Typewriter sequence f_n: the 'on' interval shrinks and sweeps across [0,1]\n"
                   "(curves offset vertically only so they don't overlap -- each is really a 0/1 step)")
axes[0].set_xlabel("x"); axes[0].set_ylabel("f_n(x)  (offset for visibility)")
axes[0].legend(fontsize=8, loc="upper right")

# Bottom: fix a handful of x values and watch f_n(x) as n grows -- it NEVER settles at any x,
# even though the interval width (and hence the L^1 norm / integral of f_n) shrinks to 0
fixed_xs = [0.1, 0.37, 0.5, 0.83]
n_range = np.arange(1, 130)
for x0 in fixed_xs:
    values = [typewriter_fn(n, np.array([x0]))[0] for n in n_range]
    axes[1].step(n_range, values, where="post", alpha=0.8, label=f"x={x0}")
axes[1].set_title("At every FIXED x, f_n(x) keeps flipping 1/0 forever as n grows\n"
                   "(no pointwise limit exists anywhere) -- yet the integral of f_n shrinks to 0")
axes[1].set_xlabel("n"); axes[1].set_ylabel("f_n(x)")
axes[1].legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

integrals = [(hi - lo) for hi, lo in (typewriter_interval(n)[::-1] for n in [1, 4, 16, 64, 256, 1024])]
print("Integral (= interval width) of f_n at n = 1, 4, 16, 64, 256, 1024:")
print("  ", [round(w, 6) for w in integrals])
print("-> shrinks to 0: f_n -> 0 in L^1 (and 'in probability'/'in measure').")
print("But no x in [0,1] has f_n(x) -> 0 or -> anything else: every x is re-hit by infinitely")
print("many n's forever, confirmed for the 4 fixed x's above never settling over 129 steps.")

## 🐍 Implementation from Scratch

We build a finite $\sigma$-algebra from scratch and verify its closure properties directly,
implement the Lebesgue integral via the "layer-cake" (horizontal-slicing) formula and check it
against `scipy.integrate.quad`, and contrast the typewriter sequence above with a function that
genuinely converges almost everywhere (but not everywhere) — the actual pattern behind "a.s."

In [ ]:
import itertools
from scipy import integrate

# --- A finite sigma-algebra built from scratch: the sigma-algebra GENERATED by a partition of
# Omega is exactly the set of all unions of partition blocks. This is the simplest possible
# concrete example of the closure rules in the Theory section, small enough to check by hand. ---
def generate_sigma_algebra_from_partition(omega, partition):
    blocks = [frozenset(b) for b in partition]
    algebra = set()
    for r in range(len(blocks) + 1):
        for combo in itertools.combinations(blocks, r):
            u = frozenset().union(*combo) if combo else frozenset()
            algebra.add(u)
    return algebra

omega = frozenset(range(4))
partition = [{0, 1}, {2}, {3}]          # 3 blocks -> a sigma-algebra of 2^3 = 8 sets
F = generate_sigma_algebra_from_partition(omega, partition)
print(f"Sigma-algebra generated by partition {partition}: {len(F)} sets (expect 2^3=8)")

# Verify the three defining closure properties directly, not just count the sets
has_omega = omega in F
has_empty = frozenset() in F
closed_complement = all(frozenset(omega - s) in F for s in F)
closed_union = all(frozenset(s1 | s2) in F for s1 in F for s2 in F)
print(f"Omega in F: {has_omega}   empty set in F: {has_empty}")
print(f"closed under complement: {closed_complement}   closed under union: {closed_union}")

# {0,2} SPLITS the block {0,1} -- it is NOT a union of whole blocks, so it is genuinely
# non-measurable with respect to THIS sigma-algebra (a finite, elementary echo of the Vitali
# set argument: some sets simply aren't assigned membership by a given F).
test_set = frozenset({0, 2})
print(f"Is {{0,2}} measurable (i.e. in F)? {test_set in F}  <- cuts across block {{0,1}}, so no")

# --- The Lebesgue integral via layer-cake / horizontal slicing: integral of f>=0 over [a,b] is
# integral over t of measure({x : f(x) > t}) dt -- literally the Theory section's formula,
# implemented directly rather than via the familiar vertical Riemann strips. ---
def trapz(y, x):
    # hand-rolled trapezoidal rule, avoiding numpy-version-dependent trapz/trapezoid naming
    dx = np.diff(x)
    return np.sum(dx * (y[:-1] + y[1:]) / 2)

def lebesgue_layer_cake(f, a, b, n_domain=100_000, n_levels=3000):
    xs = np.linspace(a, b, n_domain)
    fx = f(xs)
    ts = np.linspace(0, fx.max(), n_levels)
    domain_len = b - a
    # measure({x : f(x) > t}), estimated as (fraction of grid points above t) * domain length
    measures = np.array([np.mean(fx > t) for t in ts]) * domain_len
    return trapz(measures, ts)

print("\n--- Layer-cake Lebesgue integral vs. scipy.integrate.quad (independent ground truth) ---")
for name, f, a, b in [("x^2", lambda x: x ** 2, 0, 1),
                       ("sin(x)+1", lambda x: np.sin(x) + 1, 0, np.pi),
                       ("exp(-x^2)", lambda x: np.exp(-x ** 2), -2, 2)]:
    lc_value = lebesgue_layer_cake(f, a, b)
    quad_value, _ = integrate.quad(f, a, b)
    rel_err = abs(lc_value - quad_value) / abs(quad_value)
    print(f"  {name:10s} on [{a:.2f},{b:.2f}]: layer-cake={lc_value:.6f}  quad={quad_value:.6f}  "
          f"relative error={rel_err:.2e}")

# --- Genuinely a.e.- (but not everywhere-) convergent example, contrasted with the typewriter
# sequence's total failure above. g_n(x) = 1 on [0, 1/n], else 0: converges to 0 at EVERY x>0
# (a set of full measure), but g_n(0)=1 forever -- a single measure-zero exception. ---
def g_n(n, x):
    return np.where(x <= 1.0 / n, 1.0, 0.0)

print("\n--- g_n(x) = 1_[0, 1/n](x): converges a.e. (fails only at the single point x=0) ---")
rng = np.random.default_rng(0)
random_positive_xs = rng.uniform(1e-6, 1, size=8)   # random x, all strictly > 0 (measure-1 set)
for n in [1, 10, 100, 10_000]:
    at_zero = g_n(n, np.array([0.0]))[0]
    at_random = g_n(n, random_positive_xs).max()   # worst case among the sampled x>0
    print(f"  n={n:6d}: g_n(0)={at_zero}   max g_n(x) over 8 random x>0 = {at_random}")
print("g_n(0) never changes from 1 (the single measure-zero exception); every x>0 sampled has")
print("already reached its final value 0 by n=10_000 -- this is what 'a.e.' formally excludes:")
print("exactly the single bad point {0}, which has Lebesgue measure zero.")

## 🤖 Why This Matters for AI

Rigorous probability foundations that underpin the theorem statements in modern ML theory and
generative modeling papers. Two concrete places this shows up: the phrase "SGD converges to a
stationary point **almost surely**" is a precise measure-theoretic claim about the *set of
random seeds* for which convergence holds, not a hand-wave — we make it checkable below by
comparing a step-size schedule that satisfies the classical Robbins-Monro conditions (guaranteeing
a.s. convergence) against one that doesn't. Diffusion models (`09.06`) and score-based generative
models are literally defined as solutions to stochastic differential equations, objects that only
have a rigorous meaning once expectation is defined as a Lebesgue integral against a properly
constructed probability measure — the same machinery this notebook introduces.

In [ ]:
# "SGD converges almost surely" made checkable. The classical Robbins-Monro conditions on a
# step-size schedule a_n are sum(a_n) = infinity (steps don't shrink so fast progress halts) AND
# sum(a_n^2) < infinity (steps shrink fast enough to average out gradient noise). a_n = 1/n
# satisfies both (harmonic series diverges, sum 1/n^2 converges); a constant a_n = c does not
# (sum c^2 = infinity), so theory predicts only the decaying schedule gives a.s. convergence.
def run_sgd_many_seeds(step_fn, n_steps, noise_std, n_seeds, base_seed):
    # Minimize f(x) = 0.5*(x-3)^2 with a NOISY gradient (x-3+noise) -- a stand-in for a
    # mini-batch gradient whose noise never fully vanishes at any finite batch size. Vectorized
    # across n_seeds independent runs at once (looping n_steps times in Python, not n_seeds times).
    rng = np.random.default_rng(base_seed)
    x = np.zeros(n_seeds)
    for n in range(1, n_steps + 1):
        noisy_grad = (x - 3.0) + rng.normal(scale=noise_std, size=n_seeds)
        x = x - step_fn(n) * noisy_grad
    return x

decaying_step = lambda n: 1.0 / n     # satisfies BOTH Robbins-Monro conditions
constant_step = lambda n: 0.05         # fails the needed condition for a.s. convergence

print("Final-iterate spread (std across 150 independent seeds) as training runs LONGER:")
print(f"{'n_steps':>8} | {'decaying step 1/n: std':>24} | {'constant step 0.05: std':>24}")
for n_steps in [1000, 5000, 20_000, 80_000]:
    finals_decay = run_sgd_many_seeds(decaying_step, n_steps, 1.0, 150, base_seed=1)
    finals_const = run_sgd_many_seeds(constant_step, n_steps, 1.0, 150, base_seed=2)
    print(f"{n_steps:8d} | {finals_decay.std():24.5f} | {finals_const.std():24.5f}")

print("\nDecaying step size: spread across seeds keeps SHRINKING as n_steps grows -- consistent")
print("with almost-sure convergence to the true minimizer x*=3 (every run eventually lands there,")
print("with 'eventually' varying by seed, but the SET of seeds that fail has probability 0).")
print("Constant step size: spread PLATEAUS instead, no matter how long you train -- the iterate")
print("keeps fluctuating near x*=3 forever, never actually converging pointwise for any fixed")
print("run. This is exactly the distinction the Theory section's a.e./a.s. definition captures:")
print("'almost sure convergence' is a claim about a set of sample paths with probability exactly")
print("1, and it provably requires a shrinking step size here.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $\Omega=\{0,1,2,3\}$ and the partition into blocks $\{0,1\}, \{2\}, \{3\}$ used by
   `generate_sigma_algebra_from_partition`, list by hand every set in the generated
   $\sigma$-algebra. As a sanity check, confirm your count matches $2^{(\text{number of
   blocks})}$ — the general size of a $\sigma$-algebra generated by a finite partition. Then
   decide whether $\{0,2\}$ (a set that takes part of block $\{0,1\}$ together with all of
   $\{2\}$) belongs to this $\sigma$-algebra, and confirm your answer with the code.

<details>
<summary>💡 Solution</summary>

With 3 blocks there are $2^3=8$ sets — every union of a subcollection of the blocks, including
the empty union ($\varnothing$) and the full union ($\Omega$):
$$\varnothing,\ \{2\},\ \{3\},\ \{0,1\},\ \{2,3\},\ \{0,1,2\},\ \{0,1,3\},\ \{0,1,2,3\}.$$
That's exactly $8=2^3$, confirming the general fact. $\{0,2\}$ splits block $\{0,1\}$ down the
middle, and no union of *whole* blocks can do that, so it is **not** in the $\sigma$-algebra —
confirmed by `frozenset({0,2}) not in generate_sigma_algebra_from_partition({0,1,2,3}, [{0,1},
{2},{3}])` evaluating to `True`.

</details>

### 💭 UNDERSTAND
2. This notebook has two sequences on $[0,1]$ with Lebesgue measure whose integrals both shrink
   to $0$: the typewriter sequence `typewriter_fn(n, x)` and $g_n(x)=\mathbb{1}_{[0,1/n]}(x)$.
   Without running anything, predict for each sequence: (a) does it converge **almost
   everywhere** to $0$? (b) does it converge **in measure** (i.e. for every $\epsilon>0$,
   $\mu(\{x : |f_n(x)|>\epsilon\}) \to 0$)? Then verify both predictions.

<details>
<summary>💡 Solution</summary>

$g_n$ converges a.e. to $0$ (it fails only at the single point $x=0$, which has measure zero)
and also converges in measure ($\mu(\{x: g_n(x)>\epsilon\}) = 1/n \to 0$ for any
$\epsilon\in(0,1)$). The typewriter sequence does **not** converge a.e.: every fixed $x$ lies
inside infinitely many of the sweeping-then-shrinking intervals as $n\to\infty$, so
$f_n(x)=1$ infinitely often for *every* $x$ — the sequence never settles pointwise anywhere. It
**does**, however, converge in measure: $\mu(\{x : \text{typewriter\_fn}(n,x)=1\})$ is exactly
the interval width $1/2^{\lfloor\log_2 n\rfloor}\to 0$. So convergence in measure is strictly
weaker than a.e. convergence — the typewriter sequence is the standard example separating the
two, which is exactly why an ML theorem stating "SGD converges **almost surely**" is asserting
something strictly stronger than "converges in probability," not just using a fancier word for
the same thing.

</details>

### ✏️ DERIVE
3. Derive a formula for the typewriter sequence's interval width at index $n$ in terms of $n$'s
   binary scale, then use it to find the *smallest* $n$ for which the width first drops below
   $10^{-6}$. Verify your answer numerically with `typewriter_interval`.

<details>
<summary>💡 Solution outline</summary>

Every $n\ge 1$ decomposes uniquely as $n = 2^k+j$ with $k=\lfloor\log_2 n\rfloor$ and
$0\le j<2^k$. `typewriter_interval` slices $[0,1]$ into $2^k$ equal sub-intervals of width
$1/2^k$ at "level" $k$, and $j$ selects which one — so the width at index $n$ is exactly
$1/2^{\lfloor\log_2 n\rfloor}$, constant across the $2^k$ consecutive indices sharing that $k$,
then halving as $k$ increments. The width drops below $10^{-6}$ exactly when $2^k > 10^6$, i.e.
$k\ge 20$ (since $2^{19}=524{,}288 < 10^6 \le 2^{20}=1{,}048{,}576$). The smallest $n$ with that
$k$ is the first index at level $20$: $n=2^{20}+0=1{,}048{,}576$, giving width
$1/2^{20}\approx 9.537\times10^{-7}<10^{-6}$ — confirmed by `typewriter_interval(1_048_576)`,
while `typewriter_interval(1_048_575)` (the last index at level $19$) still has width
$1/2^{19}\approx1.907\times10^{-6}>10^{-6}$.

</details>

### 🐍 IMPLEMENT
4. Implement a shifted variant `g_n_shifted(n, x) = ` $\mathbb{1}_{[1/2,\ 1/2+1/n]}(x)$. Check
   that it converges a.e. to $0$ (i.e. at every $x\ne 1/2$) but not at $x=1/2$ itself, and
   confirm its Lebesgue integral over $[0,1]$ still shrinks to $0$ using `lebesgue_layer_cake`.

<details>
<summary>✅ How to know you're right</summary>

For any fixed $x\ne 1/2$: if $x<1/2$, `g_n_shifted(n, x)` should be $0$ for *every* $n$ (the
interval $[1/2, 1/2+1/n]$ never reaches $x$); if $x>1/2$, it should become permanently $0$ once
$n$ exceeds $1/(x-1/2)$. At $x=1/2$ exactly, it should stay $1$ for every $n$ you test — never
converging there. `lebesgue_layer_cake(lambda x: g_n_shifted(n, x), 0, 1)` should return a value
close to $1/n$ (the exact Lebesgue measure of $[1/2, 1/2+1/n]$) for each $n$, shrinking toward
$0$ as $n$ grows; a result that doesn't scale like $1/n$ means the shifted interval is off.

</details>

### 🤖 APPLY
5. The original Transformer paper (Vaswani et al., 2017) decays its learning rate like
   $c_n = c/\sqrt{n}$ after warmup. Treat $1/\sqrt{n}$ as a step-size schedule in
   `run_sgd_many_seeds` and check it against the two Robbins-Monro conditions
   ($\sum_n c_n=\infty$ and $\sum_n c_n^2<\infty$) that this notebook's a.s.-convergence claim
   for `decaying_step` relies on. Does $1/\sqrt{n}$ satisfy both?

<details>
<summary>✅ What you should observe</summary>

$\sum 1/\sqrt{n}$ diverges (even more slowly than $\sum 1/n$) — the first condition holds. But
$\sum (1/\sqrt{n})^2=\sum 1/n$ is exactly the harmonic series, which *also* diverges — so the
**second** condition fails. Running `run_sgd_many_seeds` with `step_fn = lambda n: 1.0/np.sqrt(n)`
should show the cross-seed final-iterate spread shrinking more slowly and less cleanly than the
notebook's own `decaying_step=1/n`, rather than settling into the same tidy shrinkage pattern —
a concrete, widely-used schedule that doesn't satisfy the classical sufficient condition for a.s.
convergence. It still works well in practice, but that success rests on the finite training
horizon and other implicit regularization, not on the classical asymptotic guarantee actually
applying.

</details>

### 🚀 CHALLENGE
6. This notebook's SGD experiment injects **Gaussian** noise at every step. A body of empirical
   work (e.g. Şimşekli, Sağun & Gürbüzbalaban, "A Tail-Index Analysis of Stochastic Gradient
   Noise in Deep Neural Networks," 2019) argues real deep-learning gradient noise is
   heavy-tailed rather than Gaussian. Modify (or write a variant of) `run_sgd_many_seeds` to
   inject noise from a heavy-tailed distribution instead (e.g. `rng.standard_t(df)` for a small
   `df`, rescaled to a comparable spread), and investigate whether the qualitative conclusion —
   decaying step size's cross-seed spread shrinks over training, constant step size's plateaus —
   still holds.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer confirms the injected noise really is heavier-tailed than Gaussian (e.g. via a
kurtosis check or QQ-plot) before drawing any conclusion, runs enough seeds to separate a real
behavioral change from ordinary seed variability, and reports whether occasional large jumps
(rare under Gaussian noise, common under heavy tails) visibly disrupt the decaying schedule's
convergence — for instance via outlier final iterates, or a heavier-tailed *distribution* of
final iterates even while the typical (median) behavior still shrinks. It should explicitly
connect the finding to why classical Robbins-Monro theory (a finite-variance, CLT-style
argument) is a different mathematical regime from heavy-tailed SGD noise, rather than just
reporting numbers with no interpretation.

</details>

---

## 📚 Further Reading
- Axler, *Measure, Integration & Real Analysis* — free online, modern and example-driven
- Billingsley, *Probability and Measure*, Ch. 1-2 (the classical rigorous treatment)
- Tao, *An Introduction to Measure Theory* — free lecture notes, Ch. 1

---

*Next notebook: [Stochastic Processes](02_stochastic_processes.ipynb)*